In [1]:
import xarray as xr
import pandas as pd
import glob
import os
import math

import numpy as np
import re
from joblib import Parallel, delayed
from pathlib import Path
import matplotlib.pyplot as plt

In [5]:
working_dir = '/glade/work/qingyuany/repo_data/'
case_name = "test"

In [13]:
def meta_one_hot_shot(meta, para_nm):
    meta = meta.transpose()
    meta_one_hot = pd.DataFrame(False, index=meta.index, columns=para_nm)
    for index, row in meta.iterrows():
        for r in row.values:
            meta_one_hot.loc[index, para_nm[r]] = True

    return meta_one_hot

In [14]:
class HistoryMatching:
    def __init__(self, working_dir, case_name, threshold_level = 2.0):
        self.root = Path(working_dir) / case_name
        self.tf_masks = pd.read_csv(self.root / f'tf_masks_level_{threshold_level}.csv', index_col=0)
        self.meta = pd.read_csv(self.root / 'meta.csv', index_col = 0)
        self.p_emu = xr.open_dataset(self.root / 'sampled_parameters.nc').to_dataframe()
        self.var_nm = list(self.tf_masks.columns)
        self.para_nm = list(self.p_emu.columns)
        self.meta_onehot = meta_one_hot_shot(self.meta, self.para_nm)

     def drop_by_name(self, var_to_exclude):
        var_to_drop = []
        for v in var_to_exclude:
            var_to_drop.extend([s for s in self.data_gcm.var_nm if s.startswith(v)])

        print("Drop the following variables")
        print(var_to_drop)
        self.data_emu.tf_masks = self.data_emu.tf_masks.drop(columns = var_to_drop)

        self.data_gcm.var_nm = list(self.data_emu.tf_masks.columns)
        self.var_excluded.var_dropby_name = var_to_drop

    def drop_by_n_survive(self, n_survive):
        survive_summary = self.data_emu.tf_masks.sum(axis = 0)
        self.var_excluded.var_useless = list(survive_summary[survive_summary == self.data_emu.tf_masks.shape[0]].index)
        self.var_excluded.var_tight = list(survive_summary[survive_summary < n_survive].index)

        self.data_gcm.var_nm = list(survive_summary[(survive_summary > n_survive) & (survive_summary < self.data_emu.tf_masks.shape[0])].index)
        self.data_emu.tf_masks = self.data_emu.tf_masks[self.data_gcm.var_nm]


In [8]:
test_case = HistoryMatching(working_dir, case_name)

In [12]:
test_case.meta

,SWCF_zonal_-75to-70,SWCF_zonal_-70to-65,SWCF_zonal_-65to-60,SWCF_zonal_-60to-55,SWCF_zonal_-55to-50,SWCF_zonal_-50to-45,SWCF_zonal_-45to-40,SWCF_zonal_-40to-35,SWCF_zonal_-35to-30,SWCF_zonal_-30to-25,...,FSNTOA_zonal_35to40,FSNTOA_zonal_40to45,FSNTOA_zonal_45to50,FSNTOA_zonal_50to55,FSNTOA_zonal_55to60,FSNTOA_zonal_60to65,FSNTOA_zonal_65to70,FSNTOA_zonal_70to75,SWCF_-6_-4_141_144,SWCF_7_9_235_240
0,32,24,24,24,24,8,8,8,8,8,...,8,26,8,8,8,8,8,31,23,20
1,31,8,8,8,8,29,29,29,29,29,...,26,8,26,29,29,29,31,8,16,21


In [ ]:
def meta_one_hot_shot(meta, para_nm):
    meta = meta.transpose()
    meta_one_hot = pd.DataFrame(False, index=meta.index, columns=para_nm)
    for index, row in meta.iterrows():
        for r in row.values:
            meta_one_hot.loc[index, para_nm[r]] = True

    return meta_one_hot